# 0825_peace_010_type_expert_sqrt_class_time_weight

`mapping.json`으로 각 `inspection_type`의 유효 피처만 선택하는 타입별 XGBoost 전문가 모델에 타입별 `sqrt(scale_pos_weight)`와 시간 기반 `sample_weight`를 함께 추가한 실험입니다.

- 모델·피처·파라미터·시간 분할·임계값 선택 방식은 `0825_peace_004_type_expert_walk_forward`와 동일합니다.
- 차이는 각 Fold와 최종 학습에서 타입별 Train 데이터로 `scale_pos_weight = sqrt(음성 수 / 양성 수)`를 계산하고, 동시에 타입별 Train 구간 내부 시간순 `sample_weight`를 1.0에서 2.0까지 선형 증가시킨 점뿐입니다.
- 각 Fold에서 Calibration Recall 99% 조건으로 공통·타입별 임계값을 선택하고 바로 다음 미래 구간에서 평가합니다.
- Walk-forward 완료 후 0~70% Train, 70~80% 최종 임계값 선택, 80~100% Test 평가를 동일하게 수행합니다.
- 실행 과정은 콘솔과 `docs/peace/0825_peace_010_type_expert_sqrt_class_time_weight.log`에 함께 기록합니다.


## 1. 설정, 경로 탐색과 실행 로그

노트북을 저장소 루트 또는 `notebooks/`에서 실행해도 같은 원본 파일과 로그 경로를 사용합니다.


In [1]:
import gc
import hashlib
import json
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

EXPERIMENT_ID = "0825_peace_010_type_expert_sqrt_class_time_weight"
RANDOM_STATE = 42
TARGET = "class"
TIME_COLUMN = "timestamp"
TYPE_COLUMN = "inspection_type"
RECORD_ID = "record_id"
DECISION_THRESHOLD = 0.5
MIN_RECALL = 0.99
TRAIN_END_FRACTION = 0.70
VALIDATION_END_FRACTION = 0.80
TIME_WEIGHT_MIN = 1.0
TIME_WEIGHT_MAX = 2.0
CLASS_WEIGHT_POWER = 0.5

XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "n_estimators": 400,
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "max_delta_step": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": 0,
}


def compute_sqrt_scale_pos_weight(y: pd.Series) -> dict[str, float]:
    positive = int(y.sum())
    negative = int(len(y) - positive)
    if positive == 0 or negative == 0:
        raise ValueError("scale_pos_weight는 양성과 음성이 모두 있는 Train에서만 계산할 수 있습니다.")

    raw_ratio = negative / positive
    sqrt_ratio = raw_ratio ** CLASS_WEIGHT_POWER
    return {
        "train_negative": negative,
        "train_positive": positive,
        "scale_pos_weight_raw": raw_ratio,
        "scale_pos_weight_sqrt": sqrt_ratio,
    }


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "notebooks").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("AGENTS.md가 있는 저장소 루트를 찾지 못했습니다.")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_data_pair(repo_root: Path) -> tuple[Path, Path]:
    candidates = [
        repo_root / "data" / "raw",
        repo_root.parent,
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]
    checked = set()
    for directory in candidates:
        resolved = directory.resolve()
        if resolved in checked:
            continue
        checked.add(resolved)
        data_path = resolved / "dataset.csv"
        mapping_path = resolved / "mapping.json"
        if data_path.exists() and mapping_path.exists():
            return data_path, mapping_path
    raise FileNotFoundError("dataset.csv와 mapping.json 쌍을 찾지 못했습니다.")


def make_time_sample_weight(frame: pd.DataFrame, time_column: str = TIME_COLUMN):
    if len(frame) == 0:
        raise ValueError("빈 frame에는 시간 가중치를 만들 수 없습니다.")

    timestamps = pd.to_datetime(frame[time_column], utc=True)
    timestamp_ns = timestamps.astype("int64").to_numpy(dtype=np.float64, copy=False)
    min_ns = float(timestamp_ns.min())
    max_ns = float(timestamp_ns.max())

    if max_ns == min_ns:
        weights = np.full(len(frame), TIME_WEIGHT_MIN, dtype=np.float64)
        degenerate = True
    else:
        relative_position = (timestamp_ns - min_ns) / (max_ns - min_ns)
        weights = TIME_WEIGHT_MIN + relative_position * (TIME_WEIGHT_MAX - TIME_WEIGHT_MIN)
        degenerate = False

    summary = {
        "time_weight_min": float(weights.min()),
        "time_weight_max": float(weights.max()),
        "time_weight_mean": float(weights.mean()),
        "time_weight_degenerate": degenerate,
        "train_start_time": timestamps.min(),
        "train_end_time": timestamps.max(),
    }
    return weights, summary


REPO_ROOT = find_repo_root()
DATA_PATH, MAPPING_PATH = find_data_pair(REPO_ROOT)
LOG_DIR = REPO_ROOT / "docs" / "peace"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f"{EXPERIMENT_ID}.log"

logger = logging.getLogger(EXPERIMENT_ID)
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
file_handler.setFormatter(formatter)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)
logger.propagate = False

DATA_SHA256_BEFORE = sha256_file(DATA_PATH)
MAPPING_SHA256_BEFORE = sha256_file(MAPPING_PATH)
logger.info("experiment=%s", EXPERIMENT_ID)
logger.info(
    "random_state=%d baseline_threshold=%.2f min_recall=%.2f class_weight=sqrt(negative/positive) time_weight_range=[%.1f, %.1f]",
    RANDOM_STATE,
    DECISION_THRESHOLD,
    MIN_RECALL,
    TIME_WEIGHT_MIN,
    TIME_WEIGHT_MAX,
)
logger.info("data_file=%s sha256=%s", DATA_PATH.name, DATA_SHA256_BEFORE)
logger.info("mapping_file=%s sha256=%s", MAPPING_PATH.name, MAPPING_SHA256_BEFORE)
logger.info(
    "versions python=%s pandas=%s sklearn=%s xgboost=%s",
    sys.version.split()[0], pd.__version__, sklearn.__version__, xgboost.__version__
)
logger.info("log_file=docs/peace/%s", LOG_PATH.name)
print("log saved to:", LOG_PATH.relative_to(REPO_ROOT))


2026-08-25 03:06:03,312 | INFO | experiment=0825_peace_010_type_expert_sqrt_class_time_weight


2026-08-25 03:06:03,312 | INFO | random_state=42 baseline_threshold=0.50 min_recall=0.99 class_weight=sqrt(negative/positive) time_weight_range=[1.0, 2.0]


2026-08-25 03:06:03,313 | INFO | data_file=dataset.csv sha256=53e8568743216d556856ed69b388f6750fbfa0b8c59ad31f970515ac9eb10e62


2026-08-25 03:06:03,313 | INFO | mapping_file=mapping.json sha256=3b20f440b6d9ed0baefa662e1a6f03688befbe0f28341a3b54655d3058c6e486


2026-08-25 03:06:03,313 | INFO | versions python=3.12.7 pandas=2.2.2 sklearn=1.5.1 xgboost=3.4.1


2026-08-25 03:06:03,314 | INFO | log_file=docs/peace/0825_peace_010_type_expert_sqrt_class_time_weight.log


log saved to: docs/peace/0825_peace_010_type_expert_sqrt_class_time_weight.log


## 2. 원본 데이터와 매핑 검증

첫 번째 익명 인덱스 열은 `record_id`로 이름만 바꾸며 원본 파일은 수정하지 않습니다. 중복 제거는 원인 확인 전 데이터 의미를 바꿀 수 있어 이번 베이스라인에서 수행하지 않습니다.


In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:") or source_index_column == "":
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")

with MAPPING_PATH.open(encoding="utf-8") as stream:
    feature_mapping = json.load(stream)

required_columns = {RECORD_ID, TIME_COLUMN, TYPE_COLUMN, TARGET}
missing_required = required_columns - set(raw_df.columns)
assert not missing_required, f"필수 컬럼 누락: {sorted(missing_required)}"
assert len(raw_df) == 440_274
assert raw_df[RECORD_ID].is_unique
assert set(raw_df[TARGET].unique()) == {0, 1}
assert raw_df[TARGET].value_counts().to_dict() == {0: 435_652, 1: 4_622}
assert set(raw_df[TYPE_COLUMN].unique()) == {0, 1, 2, 3, 4}
assert set(feature_mapping) == {"0", "1", "2", "3", "4"}

raw_df[TIME_COLUMN] = pd.to_datetime(raw_df[TIME_COLUMN], errors="raise", utc=True)
raw_df = raw_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)
inspection_columns = [column for column in raw_df.columns if column.startswith("inspection_feat")]
mapped_union = set().union(*(set(columns) for columns in feature_mapping.values()))
assert len(inspection_columns) == 70
assert len(mapped_union) == 65
assert mapped_union <= set(inspection_columns)

numeric_inputs = raw_df.select_dtypes(include=[np.number]).drop(columns=[TARGET, RECORD_ID])
assert np.isfinite(numeric_inputs.to_numpy()).all()

data_summary = pd.Series(
    {
        "rows": len(raw_df),
        "columns": raw_df.shape[1],
        "false_call_0": int((raw_df[TARGET] == 0).sum()),
        "real_defect_1": int((raw_df[TARGET] == 1).sum()),
        "real_defect_rate_pct": raw_df[TARGET].mean() * 100,
        "inspection_types": raw_df[TYPE_COLUMN].nunique(),
        "inspection_features": len(inspection_columns),
        "mapped_feature_union": len(mapped_union),
        "timestamp_start": raw_df[TIME_COLUMN].min(),
        "timestamp_end": raw_df[TIME_COLUMN].max(),
    },
    name="raw_data",
)
display(data_summary)
logger.info(
    "data_verified rows=%d columns=%d class_0=%d class_1=%d",
    len(raw_df), raw_df.shape[1], int((raw_df[TARGET] == 0).sum()), int((raw_df[TARGET] == 1).sum())
)


rows                                       440274
columns                                        78
false_call_0                               435652
real_defect_1                                4622
real_defect_rate_pct                     1.049801
inspection_types                                5
inspection_features                            70
mapped_feature_union                           65
timestamp_start         1970-06-23 03:58:55+00:00
timestamp_end           1970-11-02 14:21:28+00:00
Name: raw_data, dtype: object

2026-08-25 03:06:07,560 | INFO | data_verified rows=440274 columns=78 class_0=435652 class_1=4622


## 3. 타입별 유효 피처

각 전문가 모델은 공통 `meta_feat1~4`와 `mapping.json`에 명시된 해당 타입의 `inspection_feat`만 사용합니다. 타입 분리 후 상수인 `inspection_type`과 식별자·시간·타깃은 입력에서 제외합니다.


In [3]:
inspection_types = sorted(raw_df[TYPE_COLUMN].unique().tolist())
meta_columns = [column for column in raw_df.columns if column.startswith("meta_feat")]
feature_columns_by_type = {}
feature_rows = []

for inspection_type in inspection_types:
    mapped_columns = feature_mapping[str(inspection_type)]
    assert len(mapped_columns) == len(set(mapped_columns))
    assert set(mapped_columns) <= set(raw_df.columns)
    selected_columns = meta_columns + mapped_columns
    feature_columns_by_type[inspection_type] = selected_columns
    feature_rows.append(
        {
            "inspection_type": inspection_type,
            "meta_features": len(meta_columns),
            "mapped_inspection_features": len(mapped_columns),
            "total_model_features": len(selected_columns),
        }
    )

feature_summary = pd.DataFrame(feature_rows).set_index("inspection_type")
display(feature_summary)
logger.info("feature_mapping_verified=%s", feature_summary.to_dict(orient="index"))


,meta_features,mapped_inspection_features,total_model_features
inspection_type,,,
0,4,44,48
1,4,52,56
2,4,65,69
3,4,65,69
4,4,21,25


2026-08-25 03:06:07,570 | INFO | feature_mapping_verified={0: {'meta_features': 4, 'mapped_inspection_features': 44, 'total_model_features': 48}, 1: {'meta_features': 4, 'mapped_inspection_features': 52, 'total_model_features': 56}, 2: {'meta_features': 4, 'mapped_inspection_features': 65, 'total_model_features': 69}, 3: {'meta_features': 4, 'mapped_inspection_features': 65, 'total_model_features': 69}, 4: {'meta_features': 4, 'mapped_inspection_features': 21, 'total_model_features': 25}}


## 4. 시간순 Train/Validation/Test 분할

전체 행의 누적 비율에 가장 가까운 timestamp 그룹 끝을 경계로 사용합니다. 같은 timestamp 그룹은 서로 다른 구간에 들어가지 않습니다.

- 0~70%: Train
- 70~80%: Validation
- 80~100%: 최종 Test


In [4]:
timestamp_group_sizes = raw_df.groupby(TIME_COLUMN, sort=True).size()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
timestamp_index = timestamp_group_sizes.index


def boundary_at(fraction: float):
    position = int(np.searchsorted(cumulative_rows, len(raw_df) * fraction, side="left"))
    return timestamp_index[position]


train_end_time = boundary_at(TRAIN_END_FRACTION)
validation_end_time = boundary_at(VALIDATION_END_FRACTION)
train_mask = raw_df[TIME_COLUMN] <= train_end_time
validation_mask = (
    (raw_df[TIME_COLUMN] > train_end_time)
    & (raw_df[TIME_COLUMN] <= validation_end_time)
)
test_mask = raw_df[TIME_COLUMN] > validation_end_time

train_df = raw_df.loc[train_mask]
validation_df = raw_df.loc[validation_mask]
test_df = raw_df.loc[test_mask]
assert train_df[TIME_COLUMN].max() < validation_df[TIME_COLUMN].min()
assert set(train_df[TIME_COLUMN]).isdisjoint(set(validation_df[TIME_COLUMN]))
assert validation_df[TIME_COLUMN].max() < test_df[TIME_COLUMN].min()
assert set(validation_df[TIME_COLUMN]).isdisjoint(set(test_df[TIME_COLUMN]))
assert int(train_mask.sum() + validation_mask.sum() + test_mask.sum()) == len(raw_df)

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "positive_samples": int(train_df[TARGET].sum()),
            "positive_rate_pct": train_df[TARGET].mean() * 100,
            "timestamp_groups": train_df[TIME_COLUMN].nunique(),
            "start_time": train_df[TIME_COLUMN].min(),
            "end_time": train_df[TIME_COLUMN].max(),
        },
        {
            "split": "validation",
            "rows": len(validation_df),
            "positive_samples": int(validation_df[TARGET].sum()),
            "positive_rate_pct": validation_df[TARGET].mean() * 100,
            "timestamp_groups": validation_df[TIME_COLUMN].nunique(),
            "start_time": validation_df[TIME_COLUMN].min(),
            "end_time": validation_df[TIME_COLUMN].max(),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "positive_samples": int(test_df[TARGET].sum()),
            "positive_rate_pct": test_df[TARGET].mean() * 100,
            "timestamp_groups": test_df[TIME_COLUMN].nunique(),
            "start_time": test_df[TIME_COLUMN].min(),
            "end_time": test_df[TIME_COLUMN].max(),
        },
    ]
).set_index("split")
display(split_summary)
evaluation_policy = pd.Series(
    {
        "model_selection_uses_test": False,
        "threshold_selected_on_test": False,
        "fixed_test_threshold": DECISION_THRESHOLD,
    },
    name="evaluation_policy",
)
display(evaluation_policy)
logger.info("split_summary=%s", split_summary.reset_index().to_dict(orient="records"))
logger.info("test_policy model_selection=False threshold=%.2f", DECISION_THRESHOLD)


,rows,positive_samples,positive_rate_pct,timestamp_groups,start_time,end_time
split,,,,,,
train,308196,1940,0.629470,29249,1970-06-23 03:58:55+00:00,1970-10-05 00:29:59+00:00
validation,44026,357,0.810884,3400,1970-10-05 00:30:30+00:00,1970-10-13 16:54:14+00:00
test,88052,2325,2.640485,7093,1970-10-13 16:54:52+00:00,1970-11-02 14:21:28+00:00


model_selection_uses_test     False
threshold_selected_on_test    False
fixed_test_threshold            0.5
Name: evaluation_policy, dtype: object

2026-08-25 03:06:07,918 | INFO | split_summary=[{'split': 'train', 'rows': 308196, 'positive_samples': 1940, 'positive_rate_pct': 0.6294695583330089, 'timestamp_groups': 29249, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-10-05 00:29:59+0000', tz='UTC')}, {'split': 'validation', 'rows': 44026, 'positive_samples': 357, 'positive_rate_pct': 0.8108844773542907, 'timestamp_groups': 3400, 'start_time': Timestamp('1970-10-05 00:30:30+0000', tz='UTC'), 'end_time': Timestamp('1970-10-13 16:54:14+0000', tz='UTC')}, {'split': 'test', 'rows': 88052, 'positive_samples': 2325, 'positive_rate_pct': 2.640485167855358, 'timestamp_groups': 7093, 'start_time': Timestamp('1970-10-13 16:54:52+0000', tz='UTC'), 'end_time': Timestamp('1970-11-02 14:21:28+0000', tz='UTC')}]


2026-08-25 03:06:07,918 | INFO | test_policy model_selection=False threshold=0.50


## 5. Peace 실험과 동일한 평가 지표

PR-AUC, ROC-AUC, Accuracy, Precision, Recall, F1, TP/FN/FP/TN, False Call Reduction을 계산합니다. Threshold 0.5는 베이스라인 비교용이며 운영 임계값이 아닙니다.


In [5]:
def evaluate_predictions(y_true, prediction, probability):
    y_true = np.asarray(y_true, dtype=np.int8)
    prediction = np.asarray(prediction, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    has_both_classes = np.unique(y_true).size == 2
    has_positive = (tp + fn) > 0
    return {
        "rows": len(y_true),
        "positive_samples": int(y_true.sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "false_call_reduction": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "roc_auc": roc_auc_score(y_true, probability) if has_both_classes else np.nan,
        "pr_auc": average_precision_score(y_true, probability) if has_both_classes else np.nan,
    }


def evaluate_probabilities(y_true, probability, threshold=DECISION_THRESHOLD):
    probability = np.asarray(probability, dtype=np.float64)
    prediction = (probability >= threshold).astype(np.int8)
    return evaluate_predictions(y_true, prediction, probability)


def select_threshold(y_true, probability, min_recall=MIN_RECALL):
    """Recall 제약을 만족하며 False Call Reduction이 최대인 threshold를 선택한다."""
    y_true = np.asarray(y_true, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    if np.unique(y_true).size != 2:
        raise ValueError("임계값 선택에는 positive와 negative가 모두 필요합니다.")

    order = np.argsort(-probability, kind="stable")
    sorted_probability = probability[order]
    sorted_target = y_true[order]
    cumulative_tp = np.cumsum(sorted_target == 1)
    cumulative_fp = np.cumsum(sorted_target == 0)
    group_ends = np.flatnonzero(
        np.r_[sorted_probability[:-1] != sorted_probability[1:], True]
    )

    thresholds = sorted_probability[group_ends]
    tp = cumulative_tp[group_ends]
    fp = cumulative_fp[group_ends]
    total_positive = int((y_true == 1).sum())
    total_negative = int((y_true == 0).sum())
    recall = tp / total_positive
    false_call_reduction = 1.0 - (fp / total_negative)
    feasible = np.flatnonzero(recall >= min_recall)
    if feasible.size == 0:
        raise RuntimeError(f"Recall {min_recall:.2%} 조건을 만족하는 threshold가 없습니다.")

    best_local = np.lexsort(
        (thresholds[feasible], recall[feasible], false_call_reduction[feasible])
    )[-1]
    best = feasible[best_local]
    selected_threshold = float(thresholds[best])
    metrics = evaluate_probabilities(y_true, probability, selected_threshold)
    return {"threshold": selected_threshold, "min_recall": min_recall, **metrics}


# 최적화 구현이 작은 합성 예제의 완전 탐색과 같은 결과인지 검증한다.
_test_y = np.array([1, 0, 1, 0, 1, 0], dtype=np.int8)
_test_probability = np.array([0.9, 0.8, 0.7, 0.6, 0.4, 0.2])
_optimized = select_threshold(_test_y, _test_probability, min_recall=2 / 3)
_reference_rows = []
for _threshold in np.unique(_test_probability):
    _metrics = evaluate_probabilities(_test_y, _test_probability, _threshold)
    if _metrics["recall"] >= 2 / 3:
        _reference_rows.append((_metrics["false_call_reduction"], _metrics["recall"], _threshold))
_reference = max(_reference_rows)
assert np.isclose(_optimized["threshold"], _reference[2])
logger.info("threshold_selector_unit_test=PASS")

def make_preprocessor(feature_columns):
    categorical = [column for column in meta_columns if column in feature_columns]
    continuous = [column for column in feature_columns if column not in categorical]
    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore", dtype=np.float32),
                categorical,
            ),
            ("continuous", "passthrough", continuous),
        ],
        sparse_threshold=1.0,
        verbose_feature_names_out=True,
    )


2026-08-25 03:06:07,942 | INFO | threshold_selector_unit_test=PASS


## 6. 3-Fold Expanding Walk-forward 검증

첫 70% 개발 구간 안에서 Train을 누적 확장합니다. 각 Fold의 Calibration에서 임계값을 선택하고, 그 임계값을 바로 다음 미래 Evaluation에 고정 적용합니다.

각 타입 모델 학습에서는 해당 Fold Train에서 `scale_pos_weight = sqrt(음성 수 / 양성 수)`를 다시 계산하고, 같은 Train 내부 시간순 위치만 사용해 `sample_weight`를 1.0에서 2.0까지 선형 증가시킵니다.

| Fold | Train | Calibration | Evaluation |
|---|---:|---:|---:|
| Fold 1 | 0~30% | 30~40% | 40~50% |
| Fold 2 | 0~40% | 40~50% | 50~60% |
| Fold 3 | 0~50% | 50~60% | 60~70% |

Calibration과 Evaluation은 모델 학습에 사용하지 않으며, Evaluation은 임계값 선택에도 사용하지 않습니다.


In [6]:
WALK_FORWARD_SPECS = [
    {
        "fold": "fold_1",
        "train_start": 0.00,
        "train_end": 0.30,
        "calibration_start": 0.30,
        "calibration_end": 0.40,
        "evaluation_start": 0.40,
        "evaluation_end": 0.50,
    },
    {
        "fold": "fold_2",
        "train_start": 0.00,
        "train_end": 0.40,
        "calibration_start": 0.40,
        "calibration_end": 0.50,
        "evaluation_start": 0.50,
        "evaluation_end": 0.60,
    },
    {
        "fold": "fold_3",
        "train_start": 0.00,
        "train_end": 0.50,
        "calibration_start": 0.50,
        "calibration_end": 0.60,
        "evaluation_start": 0.60,
        "evaluation_end": 0.70,
    },
]

walk_forward_boundaries = {
    fraction: boundary_at(fraction)
    for fraction in [0.30, 0.40, 0.50, 0.60, 0.70]
}
walk_forward_segments = {}
walk_forward_split_rows = []

for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    train_end = walk_forward_boundaries[spec["train_end"]]
    calibration_start = walk_forward_boundaries[spec["calibration_start"]]
    calibration_end = walk_forward_boundaries[spec["calibration_end"]]
    evaluation_start = walk_forward_boundaries[spec["evaluation_start"]]
    evaluation_end = walk_forward_boundaries[spec["evaluation_end"]]

    segments = {
        "train": raw_df.loc[raw_df[TIME_COLUMN] <= train_end],
        "calibration": raw_df.loc[
            (raw_df[TIME_COLUMN] > calibration_start)
            & (raw_df[TIME_COLUMN] <= calibration_end)
        ],
        "evaluation": raw_df.loc[
            (raw_df[TIME_COLUMN] > evaluation_start)
            & (raw_df[TIME_COLUMN] <= evaluation_end)
        ],
    }
    assert segments["train"][TIME_COLUMN].max() < segments["calibration"][TIME_COLUMN].min()
    assert segments["calibration"][TIME_COLUMN].max() < segments["evaluation"][TIME_COLUMN].min()
    assert set(segments["train"][TIME_COLUMN]).isdisjoint(segments["calibration"][TIME_COLUMN])
    assert set(segments["calibration"][TIME_COLUMN]).isdisjoint(segments["evaluation"][TIME_COLUMN])
    walk_forward_segments[fold_name] = segments

    for segment_name, frame in segments.items():
        walk_forward_split_rows.append(
            {
                "fold": fold_name,
                "segment": segment_name,
                "rows": len(frame),
                "positive_samples": int(frame[TARGET].sum()),
                "positive_rate_pct": frame[TARGET].mean() * 100,
                "timestamp_groups": frame[TIME_COLUMN].nunique(),
                "start_time": frame[TIME_COLUMN].min(),
                "end_time": frame[TIME_COLUMN].max(),
            }
        )

walk_forward_split_summary = pd.DataFrame(walk_forward_split_rows).set_index(
    ["fold", "segment"]
)
display(walk_forward_split_summary)
logger.info(
    "walk_forward_split_summary=%s",
    walk_forward_split_summary.reset_index().to_dict(orient="records"),
)


rows  positive_samples  positive_rate_pct  \
fold   segment                                                    
fold_1 train        132137              1223           0.925555   
       calibration   43979               200           0.454763   
       evaluation    44040               326           0.740236   
fold_2 train        176116              1423           0.807990   
       calibration   44040               326           0.740236   
       evaluation    44187               152           0.343993   
fold_3 train        220156              1749           0.794437   
       calibration   44187               152           0.343993   
       evaluation    43853                39           0.088933   

                    timestamp_groups                start_time  \
fold   segment                                                   
fold_1 train                   15230 1970-06-23 03:58:55+00:00   
       calibration              1251 1970-08-18 06:51:41+00:00   
       evaluation               5415 1970-08-21 23:33:55+00:00   
fold_2 train                   16481 1970-06-23 03:58:55+00:00   
       calibration              5415 1970-08-21 23:33:55+00:00   
       evaluation               4167 1970-09-15 06:47:13+00:00   
fold_3 train                   21896 1970-06-23 03:58:55+00:00   
       calibration              4167 1970-09-15 06:47:13+00:00   
       evaluation               3186 1970-09-28 05:11:13+00:00   

                                    end_time  
fold   segment                                
fold_1 train       1970-08-18 06:51:10+00:00  
       calibration 1970-08-21 23:32:59+00:00  
       evaluation  1970-09-15 06:46:33+00:00  
fold_2 train       1970-08-21 23:32:59+00:00  
       calibration 1970-09-15 06:46:33+00:00  
       evaluation  1970-09-28 05:10:37+00:00  
fold_3 train       1970-09-15 06:46:33+00:00  
       calibration 1970-09-28 05:10:37+00:00  
       evaluation  1970-10-05 00:29:59+00:00

2026-08-25 03:06:08,606 | INFO | walk_forward_split_summary=[{'fold': 'fold_1', 'segment': 'train', 'rows': 132137, 'positive_samples': 1223, 'positive_rate_pct': 0.9255545380930399, 'timestamp_groups': 15230, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-08-18 06:51:10+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'calibration', 'rows': 43979, 'positive_samples': 200, 'positive_rate_pct': 0.4547625002842266, 'timestamp_groups': 1251, 'start_time': Timestamp('1970-08-18 06:51:41+0000', tz='UTC'), 'end_time': Timestamp('1970-08-21 23:32:59+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'evaluation', 'rows': 44040, 'positive_samples': 326, 'positive_rate_pct': 0.740236148955495, 'timestamp_groups': 5415, 'start_time': Timestamp('1970-08-21 23:33:55+0000', tz='UTC'), 'end_time': Timestamp('1970-09-15 06:46:33+0000', tz='UTC')}, {'fold': 'fold_2', 'segment': 'train', 'rows': 176116, 'positive_samples': 1423, 'positive_rate_pct': 0.807990188

In [7]:
def fit_type_experts_for_fold(train_frame, calibration_frame, evaluation_frame, fold_name):
    calibration_probability = pd.Series(np.nan, index=calibration_frame.index, dtype="float64")
    evaluation_probability = pd.Series(np.nan, index=evaluation_frame.index, dtype="float64")
    training_rows = []

    for inspection_type in inspection_types:
        feature_columns = feature_columns_by_type[inspection_type]
        type_train = train_frame.loc[train_frame[TYPE_COLUMN] == inspection_type]
        type_calibration = calibration_frame.loc[calibration_frame[TYPE_COLUMN] == inspection_type]
        type_evaluation = evaluation_frame.loc[evaluation_frame[TYPE_COLUMN] == inspection_type]
        y_train = type_train[TARGET].astype("int8")
        class_weight_summary = compute_sqrt_scale_pos_weight(y_train)
        sample_weight, time_weight_summary = make_time_sample_weight(type_train)

        assert len(type_train) > 0 and len(type_calibration) > 0 and len(type_evaluation) > 0
        assert y_train.nunique() == 2, f"fold={fold_name} type={inspection_type} Train에 두 클래스가 없습니다."
        logger.info(
            "walk_forward_fit_start fold=%s type=%d train_rows=%d train_positive=%d calibration_rows=%d calibration_positive=%d evaluation_rows=%d evaluation_positive=%d raw_features=%d scale_pos_weight_raw=%.6f scale_pos_weight_sqrt=%.6f weight_min=%.6f weight_max=%.6f weight_mean=%.6f weight_degenerate=%s",
            fold_name,
            inspection_type,
            len(type_train),
            int(y_train.sum()),
            len(type_calibration),
            int(type_calibration[TARGET].sum()),
            len(type_evaluation),
            int(type_evaluation[TARGET].sum()),
            len(feature_columns),
            class_weight_summary["scale_pos_weight_raw"],
            class_weight_summary["scale_pos_weight_sqrt"],
            time_weight_summary["time_weight_min"],
            time_weight_summary["time_weight_max"],
            time_weight_summary["time_weight_mean"],
            time_weight_summary["time_weight_degenerate"],
        )

        preprocessor = make_preprocessor(feature_columns)
        X_train = preprocessor.fit_transform(type_train[feature_columns])
        X_calibration = preprocessor.transform(type_calibration[feature_columns])
        X_evaluation = preprocessor.transform(type_evaluation[feature_columns])

        model = XGBClassifier(
            **XGB_PARAMS,
            scale_pos_weight=class_weight_summary["scale_pos_weight_sqrt"],
        )
        model.fit(X_train, y_train, sample_weight=sample_weight, verbose=False)
        calibration_probability.loc[type_calibration.index] = model.predict_proba(
            X_calibration
        )[:, 1]
        evaluation_probability.loc[type_evaluation.index] = model.predict_proba(
            X_evaluation
        )[:, 1]

        training_rows.append(
            {
                "fold": fold_name,
                "inspection_type": inspection_type,
                "train_rows": len(type_train),
                "train_positive": int(y_train.sum()),
                "calibration_rows": len(type_calibration),
                "calibration_positive": int(type_calibration[TARGET].sum()),
                "evaluation_rows": len(type_evaluation),
                "evaluation_positive": int(type_evaluation[TARGET].sum()),
                "raw_features": len(feature_columns),
                "encoded_features": X_train.shape[1],
                **class_weight_summary,
                **time_weight_summary,
            }
        )
        logger.info("walk_forward_fit_done fold=%s type=%d", fold_name, inspection_type)
        del preprocessor, model, X_train, X_calibration, X_evaluation, sample_weight
        gc.collect()

    assert calibration_probability.notna().all()
    assert evaluation_probability.notna().all()
    return calibration_probability, evaluation_probability, training_rows


walk_forward_threshold_rows = []
walk_forward_metric_rows = []
walk_forward_type_evaluation_rows = []
walk_forward_training_rows = []

for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    segments = walk_forward_segments[fold_name]
    calibration_frame = segments["calibration"]
    evaluation_frame = segments["evaluation"]
    calibration_probability, evaluation_probability, training_rows = (
        fit_type_experts_for_fold(
            segments["train"], calibration_frame, evaluation_frame, fold_name
        )
    )
    walk_forward_training_rows.extend(training_rows)

    global_selection = select_threshold(
        calibration_frame[TARGET], calibration_probability, min_recall=MIN_RECALL
    )
    walk_forward_threshold_rows.append(
        {"fold": fold_name, "scope": "global", **global_selection}
    )

    thresholds_by_type_fold = {}
    type_evaluation_prediction = pd.Series(
        np.nan, index=evaluation_frame.index, dtype="float64"
    )
    for inspection_type in inspection_types:
        type_calibration = calibration_frame.loc[
            calibration_frame[TYPE_COLUMN] == inspection_type
        ]
        type_calibration_probability = calibration_probability.loc[
            type_calibration.index
        ]
        selection = select_threshold(
            type_calibration[TARGET],
            type_calibration_probability,
            min_recall=MIN_RECALL,
        )
        threshold = selection["threshold"]
        thresholds_by_type_fold[inspection_type] = threshold
        walk_forward_threshold_rows.append(
            {
                "fold": fold_name,
                "scope": f"type_{inspection_type}",
                **selection,
            }
        )

        type_evaluation = evaluation_frame.loc[
            evaluation_frame[TYPE_COLUMN] == inspection_type
        ]
        type_evaluation_probability = evaluation_probability.loc[type_evaluation.index]
        type_prediction = (type_evaluation_probability >= threshold).astype("int8")
        type_evaluation_prediction.loc[type_evaluation.index] = type_prediction
        type_metrics = evaluate_predictions(
            type_evaluation[TARGET], type_prediction, type_evaluation_probability
        )
        type_metrics.update(
            {
                "fold": fold_name,
                "inspection_type": inspection_type,
                "threshold": threshold,
            }
        )
        walk_forward_type_evaluation_rows.append(type_metrics)

    strategy_metrics = {
        "fixed_0.5": evaluate_probabilities(
            evaluation_frame[TARGET], evaluation_probability, DECISION_THRESHOLD
        ),
        "global_threshold": evaluate_probabilities(
            evaluation_frame[TARGET],
            evaluation_probability,
            global_selection["threshold"],
        ),
        "type_specific_thresholds": evaluate_predictions(
            evaluation_frame[TARGET],
            type_evaluation_prediction,
            evaluation_probability,
        ),
    }
    for strategy, metrics in strategy_metrics.items():
        walk_forward_metric_rows.append(
            {"fold": fold_name, "strategy": strategy, **metrics}
        )

    logger.info(
        "walk_forward_fold_done fold=%s global_threshold=%.8f metrics=%s",
        fold_name,
        global_selection["threshold"],
        strategy_metrics,
    )

walk_forward_threshold_summary = pd.DataFrame(walk_forward_threshold_rows).set_index(
    ["fold", "scope"]
)
walk_forward_evaluation_metrics = pd.DataFrame(walk_forward_metric_rows).set_index(
    ["fold", "strategy"]
)
walk_forward_type_evaluation = pd.DataFrame(
    walk_forward_type_evaluation_rows
).set_index(["fold", "inspection_type"])
walk_forward_training_summary = pd.DataFrame(walk_forward_training_rows).set_index(
    ["fold", "inspection_type"]
)


2026-08-25 03:06:08,633 | INFO | walk_forward_fit_start fold=fold_1 type=0 train_rows=28277 train_positive=32 calibration_rows=8408 calibration_positive=11 evaluation_rows=6496 evaluation_positive=50 raw_features=48 scale_pos_weight_raw=882.656250 scale_pos_weight_sqrt=29.709531 weight_min=1.000000 weight_max=2.000000 weight_mean=1.516054 weight_degenerate=False


2026-08-25 03:06:09,136 | INFO | walk_forward_fit_done fold=fold_1 type=0


2026-08-25 03:06:09,169 | INFO | walk_forward_fit_start fold=fold_1 type=1 train_rows=22698 train_positive=269 calibration_rows=3868 calibration_positive=20 evaluation_rows=2618 evaluation_positive=186 raw_features=56 scale_pos_weight_raw=83.379182 scale_pos_weight_sqrt=9.131220 weight_min=1.000000 weight_max=2.000000 weight_mean=1.475024 weight_degenerate=False


2026-08-25 03:06:09,543 | INFO | walk_forward_fit_done fold=fold_1 type=1


2026-08-25 03:06:09,579 | INFO | walk_forward_fit_start fold=fold_1 type=2 train_rows=42288 train_positive=408 calibration_rows=16448 calibration_positive=92 evaluation_rows=18964 evaluation_positive=49 raw_features=69 scale_pos_weight_raw=102.647059 scale_pos_weight_sqrt=10.131488 weight_min=1.000000 weight_max=2.000000 weight_mean=1.593579 weight_degenerate=False


...

2026-08-25 03:06:16,493 | INFO | walk_forward_fit_start fold=fold_3 type=3 train_rows=67320 train_positive=622 calibration_rows=20747 calibration_positive=23 evaluation_rows=12673 evaluation_positive=3 raw_features=69 scale_pos_weight_raw=107.231511 scale_pos_weight_sqrt=10.355265 weight_min=1.000000 weight_max=2.000000 weight_mean=1.525556 weight_degenerate=False


2026-08-25 03:06:17,257 | INFO | walk_forward_fit_done fold=fold_3 type=3


2026-08-25 03:06:17,285 | INFO | walk_forward_fit_start fold=fold_3 type=4 train_rows=2771 train_positive=10 calibration_rows=698 calibration_positive=3 evaluation_rows=344 evaluation_positive=0 raw_features=25 scale_pos_weight_raw=276.100000 scale_pos_weight_sqrt=16.616257 weight_min=1.000000 weight_max=2.000000 weight_mean=1.526082 weight_degenerate=False


2026-08-25 03:06:17,390 | INFO | walk_forward_fit_done fold=fold_3 type=4


2026-08-25 03:06:17,666 | INFO | walk_forward_fold_done fold=fold_3 global_threshold=0.00012243 metrics={'fixed_0.5': {'rows': 43853, 'positive_samples': 39, 'tn': 43219, 'fp': 595, 'fn': 26, 'tp': 13, 'accuracy': 0.9858390532004652, 'precision': 0.02138157894736842, 'recall': 0.3333333333333333, 'false_call_reduction': 0.9864198657963208, 'f1': 0.0401854714064915, 'roc_auc': 0.9070359199085176, 'pr_auc': 0.014978358720840535}, 'global_threshold': {'rows': 43853, 'positive_samples': 39, 'tn': 4270, 'fp': 39544, 'fn': 0, 'tp': 39, 'accuracy': 0.09826009623058855, 'precision': 0.0009852714549175151, 'recall': 1.0, 'false_call_reduction': 0.09745743369699184, 'f1': 0.001968603301196305, 'roc_auc': 0.9070359199085176, 'pr_auc': 0.014978358720840535}, 'type_specific_thresholds': {'rows': 43853, 'positive_samples': 39, 'tn': 13694, 'fp': 30120, 'fn': 2, 'tp': 37, 'accuracy': 0.31311426812304743, 'precision': 0.0012269124912955532, 'recall': 0.9487179487179487, 'false_call_reduction': 0.31254

## 7. Walk-forward 미래 Evaluation 결과

공통·타입별 임계값은 각 Fold의 Calibration에서만 선택됐습니다. 아래 지표는 임계값 선택에 사용하지 않은 바로 다음 미래 Evaluation 결과입니다.

In [8]:
display(
    walk_forward_threshold_summary[
        ["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn"]
    ]
)
display(
    walk_forward_evaluation_metrics[
        [
            "positive_samples",
            "pr_auc",
            "precision",
            "recall",
            "false_call_reduction",
            "f1",
            "tp",
            "fn",
            "fp",
            "tn",
        ]
    ]
)
display(
    walk_forward_type_evaluation[
        [
            "threshold",
            "positive_samples",
            "pr_auc",
            "recall",
            "false_call_reduction",
            "tp",
            "fn",
        ]
    ]
)

walk_forward_strategy_summary = (
    walk_forward_evaluation_metrics.reset_index()
    .groupby("strategy")
    .agg(
        folds=("fold", "nunique"),
        mean_pr_auc=("pr_auc", "mean"),
        mean_recall=("recall", "mean"),
        min_recall=("recall", "min"),
        recall_99_folds=("recall", lambda values: int((values >= MIN_RECALL).sum())),
        mean_false_call_reduction=("false_call_reduction", "mean"),
        min_false_call_reduction=("false_call_reduction", "min"),
        total_tp=("tp", "sum"),
        total_fn=("fn", "sum"),
    )
)
display(walk_forward_strategy_summary)
display(walk_forward_training_summary)
logger.info(
    "walk_forward_strategy_summary=%s",
    walk_forward_strategy_summary.to_dict(orient="index"),
)


threshold  positive_samples    recall  false_call_reduction  \
fold   scope                                                                 
fold_1 global   0.000083               200  0.990000              0.067270   
       type_0   0.001377                11  1.000000              0.386805   
       type_1   0.001119                20  1.000000              0.824844   
       type_2   0.000016                92  1.000000              0.003302   
       type_3   0.000235                73  1.000000              0.227729   
       type_4   0.001159                 4  1.000000              0.856971   
fold_2 global   0.000596               326  0.990798              0.287048   
       type_0   0.000596                50  1.000000              0.474868   
       type_1   0.000451               186  0.994624              0.259457   
       type_2   0.001695                49  1.000000              0.482686   
       type_3   0.041085                39  1.000000              0.768752   
       type_4   0.000104                 2  1.000000              0.027864   
fold_3 global   0.000122               152  0.993421              0.152788   
       type_0   0.000112                14  1.000000              0.471073   
       type_1   0.001251                80  1.000000              0.223751   
       type_2   0.000382                32  1.000000              0.184555   
       type_3   0.000830                23  1.000000              0.711494   
       type_4   0.008801                 3  1.000000              0.815827   

                tp  fn  
fold   scope            
fold_1 global  198   2  
       type_0   11   0  
       type_1   20   0  
       type_2   92   0  
       type_3   73   0  
       type_4    4   0  
fold_2 global  323   3  
       type_0   50   0  
       type_1  185   1  
       type_2   49   0  
       type_3   39   0  
       type_4    2   0  
fold_3 global  151   1  
       type_0   14   0  
       type_1   80   0  
       type_2   32   0  
       type_3   23   0  
       type_4    3   0

positive_samples    pr_auc  precision  \
fold   strategy                                                          
fold_1 fixed_0.5                              326  0.205859   0.170929   
       global_threshold                       326  0.205859   0.007754   
       type_specific_thresholds               326  0.205859   0.008684   
fold_2 fixed_0.5                              152  0.032963   0.063025   
       global_threshold                       152  0.032963   0.005437   
       type_specific_thresholds               152  0.032963   0.009040   
fold_3 fixed_0.5                               39  0.014978   0.021382   
       global_threshold                        39  0.014978   0.000985   
       type_specific_thresholds                39  0.014978   0.001227   

                                   recall  false_call_reduction        f1  \
fold   strategy                                                             
fold_1 fixed_0.5                 0.389571              0.985908  0.237605   
       global_threshold          1.000000              0.045683  0.015389   
       type_specific_thresholds  0.978528              0.166949  0.017215   
fold_2 fixed_0.5                 0.197368              0.989872  0.095541   
       global_threshold          0.960526              0.393460  0.010812   
       type_specific_thresholds  0.835526              0.683865  0.017887   
fold_3 fixed_0.5                 0.333333              0.986420  0.040185   
       global_threshold          1.000000              0.097457  0.001969   
       type_specific_thresholds  0.948718              0.312549  0.002451   

                                  tp   fn     fp     tn  
fold   strategy                                          
fold_1 fixed_0.5                 127  199    616  43098  
       global_threshold          326    0  41717   1997  
       type_specific_thresholds  319    7  36416   7298  
fold_2 fixed_0.5                  30  122    446  43589  
       global_threshold          146    6  26709  17326  
       type_specific_thresholds  127   25  13921  30114  
fold_3 fixed_0.5                  13   26    595  43219  
       global_threshold           39    0  39544   4270  
       type_specific_thresholds   37    2  30120  13694

threshold  positive_samples    pr_auc    recall  \
fold   inspection_type                                                    
fold_1 0                 0.001377                50  0.522172  0.960000   
       1                 0.001119               186  0.215014  0.983871   
       2                 0.000016                49  0.455802  1.000000   
       3                 0.000235                39  0.639950  1.000000   
       4                 0.001159                 2  0.005584  0.000000   
fold_2 0                 0.000596                14  0.005003  0.642857   
       1                 0.000451                80  0.148913  1.000000   
       2                 0.001695                32  0.033425  0.937500   
       3                 0.041085                23  0.002887  0.217391   
       4                 0.000104                 3  0.062757  1.000000   
fold_3 0                 0.000112                 4  0.006797  1.000000   
       1                 0.001251                25  0.015606  1.000000   
       2                 0.000382                 7  0.071040  0.857143   
       3                 0.000830                 3  0.174999  0.666667   
       4                 0.008801                 0       NaN       NaN   

                        false_call_reduction   tp  fn  
fold   inspection_type                                 
fold_1 0                            0.544213   48   2  
       1                            0.409951  183   3  
       2                            0.000106   49   0  
       3                            0.169509   39   0  
       4                            0.455108    0   2  
fold_2 0                            0.611415    9   5  
       1                            0.421404   80   0  
       2                            0.496553   30   2  
       3                            0.878933    5  18  
       4                            0.014388    3   0  
fold_3 0                            0.197306    4   0  
       1                            0.101971   25   0  
       2                            0.151686    6   1  
       3                            0.665114    2   1  
       4                            0.799419    0   0

,folds,mean_pr_auc,mean_recall,min_recall,recall_99_folds,mean_false_call_reduction,min_false_call_reduction,total_tp,total_fn
strategy,,,,,,,,,
fixed_0.5,3,0.0846,0.306757,0.197368,0,0.987400,0.985908,170,347
global_threshold,3,0.0846,0.986842,0.960526,2,0.178867,0.045683,511,6
type_specific_thresholds,3,0.0846,0.920924,0.835526,0,0.387787,0.166949,483,34


train_rows  train_positive  calibration_rows  \
fold   inspection_type                                                 
fold_1 0                     28277              32              8408   
       1                     22698             269              3868   
       2                     42288             408             16448   
       3                     37264             510             14419   
       4                      1610               4               836   
fold_2 0                     36685              43              6496   
       1                     26566             289              2618   
       2                     58736             500             18964   
       3                     51683             583             15637   
       4                      2446               8               325   
fold_3 0                     43181              93              8985   
       1                     29184             475              5023   
       2                     77700             549              8734   
       3                     67320             622             20747   
       4                      2771              10               698   

                        calibration_positive  evaluation_rows  \
fold   inspection_type                                          
fold_1 0                                  11             6496   
       1                                  20             2618   
       2                                  92            18964   
       3                                  73            15637   
       4                                   4              325   
fold_2 0                                  50             8985   
       1                                 186             5023   
       2                                  49             8734   
       3                                  39            20747   
       4                                   2              698   
fold_3 0                                  14            12107   
       1                                  80             4693   
       2                                  32            14036   
       3                                  23            12673   
       4                                   3              344   

                        evaluation_positive  raw_features  encoded_features  \
fold   inspection_type                                                        
fold_1 0                                 50            48                80   
       1                                186            56               106   
       2                                 49            69               114   
       3                                 39            69               107   
       4                                  2            25                47   
fold_2 0                                 14            48                82   
       1                                 80            56               110   
       2                                 32            69               114   
       3                                 23            69               107   
       4                                  3            25                47   
fold_3 0                                  4            48                84   
       1                                 25            56               111   
       2                                  7            69               115   
       3                                  3            69               108   
       4                                  0            25                50   

                        train_negative  scale_pos_weight_raw  \
fold   inspection_type                                         
fold_1 0                         28245            882.656250   
       1                         22429             83.379182   
       2                         41880            102.647059   
       3                     

2026-08-25 03:06:17,691 | INFO | walk_forward_strategy_summary={'fixed_0.5': {'folds': 3, 'mean_pr_auc': 0.08460025866097819, 'mean_recall': 0.30675743551106804, 'min_recall': 0.19736842105263158, 'recall_99_folds': 0, 'mean_false_call_reduction': 0.9873999877993054, 'min_false_call_reduction': 0.9859084046300957, 'total_tp': 170, 'total_fn': 347}, 'global_threshold': {'folds': 3, 'mean_pr_auc': 0.08460025866097819, 'mean_recall': 0.9868421052631579, 'min_recall': 0.9605263157894737, 'recall_99_folds': 2, 'mean_false_call_reduction': 0.17886682891480596, 'min_false_call_reduction': 0.045683305119641304, 'total_tp': 511, 'total_fn': 6}, 'type_specific_thresholds': {'folds': 3, 'mean_pr_auc': 0.08460025866097819, 'mean_recall': 0.9209239572897951, 'min_recall': 0.8355263157894737, 'recall_99_folds': 0, 'mean_false_call_reduction': 0.3877874704557536, 'min_false_call_reduction': 0.16694880358695155, 'total_tp': 483, 'total_fn': 34}}


## 8. 최종 타입별 전문가 모델 5개 학습

각 타입에서 전처리기는 Train에만 `fit`합니다. 최종 0~70% Train에서 타입별 `scale_pos_weight = sqrt(음성 수 / 양성 수)`와 시간순 `sample_weight 1.0→2.0`를 함께 적용해 학습하며, 리샘플링은 적용하지 않습니다.


In [9]:
pooled_probability = pd.Series(np.nan, index=validation_df.index, dtype="float64")
models_by_type = {}
preprocessors_by_type = {}
type_metric_rows = []
training_rows = []

for inspection_type in inspection_types:
    feature_columns = feature_columns_by_type[inspection_type]
    type_train = train_df.loc[train_df[TYPE_COLUMN] == inspection_type]
    type_validation = validation_df.loc[validation_df[TYPE_COLUMN] == inspection_type]
    y_train = type_train[TARGET].astype("int8")
    y_validation = type_validation[TARGET].astype("int8")

    assert len(type_train) > 0 and len(type_validation) > 0
    assert y_train.nunique() == 2, f"type={inspection_type} Train에 두 클래스가 없습니다."
    class_weight_summary = compute_sqrt_scale_pos_weight(y_train)
    sample_weight, time_weight_summary = make_time_sample_weight(type_train)
    logger.info(
        "model_fit_start type=%d train_rows=%d train_positive=%d valid_rows=%d valid_positive=%d raw_features=%d scale_pos_weight_raw=%.6f scale_pos_weight_sqrt=%.6f weight_min=%.6f weight_max=%.6f weight_mean=%.6f weight_degenerate=%s",
        inspection_type,
        len(type_train),
        int(y_train.sum()),
        len(type_validation),
        int(y_validation.sum()),
        len(feature_columns),
        class_weight_summary["scale_pos_weight_raw"],
        class_weight_summary["scale_pos_weight_sqrt"],
        time_weight_summary["time_weight_min"],
        time_weight_summary["time_weight_max"],
        time_weight_summary["time_weight_mean"],
        time_weight_summary["time_weight_degenerate"],
    )

    preprocessor = make_preprocessor(feature_columns)
    X_train = preprocessor.fit_transform(type_train[feature_columns])
    X_validation = preprocessor.transform(type_validation[feature_columns])
    assert np.isfinite(X_train.data if hasattr(X_train, "data") else X_train).all()
    assert np.isfinite(X_validation.data if hasattr(X_validation, "data") else X_validation).all()

    model = XGBClassifier(
        **XGB_PARAMS,
        scale_pos_weight=class_weight_summary["scale_pos_weight_sqrt"],
    )
    model.fit(X_train, y_train, sample_weight=sample_weight, verbose=False)
    probability = model.predict_proba(X_validation)[:, 1]
    pooled_probability.loc[type_validation.index] = probability

    metrics = evaluate_probabilities(y_validation, probability)
    metrics["inspection_type"] = inspection_type
    type_metric_rows.append(metrics)
    training_rows.append(
        {
            "inspection_type": inspection_type,
            "train_rows": len(type_train),
            "train_positive": int(y_train.sum()),
            "validation_rows": len(type_validation),
            "validation_positive": int(y_validation.sum()),
            "raw_features": len(feature_columns),
            "encoded_features": X_train.shape[1],
            "trees": model.n_estimators,
            **class_weight_summary,
            **time_weight_summary,
        }
    )
    models_by_type[inspection_type] = model
    preprocessors_by_type[inspection_type] = preprocessor
    logger.info(
        "model_fit_done type=%d pr_auc=%.6f recall=%.6f fcr=%.6f tp=%d fn=%d",
        inspection_type,
        metrics["pr_auc"],
        metrics["recall"],
        metrics["false_call_reduction"],
        metrics["tp"],
        metrics["fn"],
    )
    del X_train, X_validation, probability, sample_weight
    gc.collect()

assert pooled_probability.notna().all()
pooled_metrics = pd.Series(
    evaluate_probabilities(validation_df[TARGET], pooled_probability),
    name="type_expert_validation",
)
type_metrics = pd.DataFrame(type_metric_rows).set_index("inspection_type")
training_summary = pd.DataFrame(training_rows).set_index("inspection_type")
logger.info("pooled_validation_metrics=%s", pooled_metrics.to_dict())


2026-08-25 03:06:17,749 | INFO | model_fit_start type=0 train_rows=64273 train_positive=111 valid_rows=13289 valid_positive=12 raw_features=48 scale_pos_weight_raw=578.036036 scale_pos_weight_sqrt=24.042380 weight_min=1.000000 weight_max=2.000000 weight_mean=1.571337 weight_degenerate=False


2026-08-25 03:06:18,582 | INFO | model_fit_done type=0 pr_auc=0.006495 recall=0.000000 fcr=0.999849 tp=0 fn=12


2026-08-25 03:06:18,618 | INFO | model_fit_start type=1 train_rows=38900 train_positive=580 valid_rows=6422 valid_positive=224 raw_features=56 scale_pos_weight_raw=66.068966 scale_pos_weight_sqrt=8.128282 weight_min=1.000000 weight_max=2.000000 weight_mean=1.482090 weight_degenerate=False


2026-08-25 03:06:19,173 | INFO | model_fit_done type=1 pr_auc=0.706346 recall=0.875000 fcr=0.926751 tp=196 fn=28


2026-08-25 03:06:19,213 | INFO | model_fit_start type=2 train_rows=100470 train_positive=588 valid_rows=7161 valid_positive=27 raw_features=69 scale_pos_weight_raw=169.867347 scale_pos_weight_sqrt=13.033317 weight_min=1.000000 weight_max=2.000000 weight_mean=1.571788 weight_degenerate=False


...

2026-08-25 03:06:20,326 | INFO | model_fit_start type=3 train_rows=100740 train_positive=648 valid_rows=16252 valid_positive=21 raw_features=69 scale_pos_weight_raw=154.462963 scale_pos_weight_sqrt=12.428313 weight_min=1.000000 weight_max=2.000000 weight_mean=1.580274 weight_degenerate=False


2026-08-25 03:06:21,564 | INFO | model_fit_done type=3 pr_auc=0.074823 recall=0.476190 fcr=0.992915 tp=10 fn=11


2026-08-25 03:06:21,595 | INFO | model_fit_start type=4 train_rows=3813 train_positive=13 valid_rows=902 valid_positive=73 raw_features=25 scale_pos_weight_raw=292.307692 scale_pos_weight_sqrt=17.097008 weight_min=1.000000 weight_max=2.000000 weight_mean=1.556784 weight_degenerate=False


2026-08-25 03:06:21,727 | INFO | model_fit_done type=4 pr_auc=0.071318 recall=0.013699 fcr=0.998794 tp=1 fn=72


2026-08-25 03:06:21,786 | INFO | pooled_validation_metrics={'rows': 44026.0, 'positive_samples': 357.0, 'tn': 43079.0, 'fp': 590.0, 'fn': 135.0, 'tp': 222.0, 'accuracy': 0.9835324580929451, 'precision': 0.2733990147783251, 'recall': 0.6218487394957983, 'false_call_reduction': 0.9864892715656415, 'f1': 0.3798118049615056, 'roc_auc': 0.8860953160947909, 'pr_auc': 0.44926386830748244}


## 9. 최종 Validation 결과


In [10]:
count_columns = ["rows", "positive_samples", "tn", "fp", "fn", "tp"]
type_metrics[count_columns] = type_metrics[count_columns].astype("int64")
display(pooled_metrics)
display(
    type_metrics[
        [
            "rows",
            "positive_samples",
            "pr_auc",
            "roc_auc",
            "accuracy",
            "precision",
            "recall",
            "false_call_reduction",
            "f1",
            "tp",
            "fn",
            "fp",
            "tn",
        ]
    ]
)
display(training_summary)


rows                    44026.000000
positive_samples          357.000000
tn                      43079.000000
fp                        590.000000
fn                        135.000000
tp                        222.000000
accuracy                    0.983532
precision                   0.273399
recall                      0.621849
false_call_reduction        0.986489
f1                          0.379812
roc_auc                     0.886095
pr_auc                      0.449264
Name: type_expert_validation, dtype: float64

,rows,positive_samples,pr_auc,roc_auc,accuracy,precision,recall,false_call_reduction,f1,tp,fn,fp,tn
inspection_type,,,,,,,,,,,,,
0,13289,12,0.006495,0.872543,0.998946,0.000000,0.000000,0.999849,0.000000,0,12,2,13275
1,6422,224,0.706346,0.958480,0.924945,0.301538,0.875000,0.926751,0.448513,196,28,454,5744
2,7161,27,0.409729,0.950275,0.995811,0.454545,0.555556,0.997477,0.500000,15,12,18,7116
3,16252,21,0.074823,0.933883,0.992247,0.080000,0.476190,0.992915,0.136986,10,11,115,16116
4,902,73,0.071318,0.309483,0.919069,0.500000,0.013699,0.998794,0.026667,1,72,1,828


,train_rows,train_positive,validation_rows,validation_positive,raw_features,encoded_features,trees,train_negative,scale_pos_weight_raw,scale_pos_weight_sqrt,time_weight_min,time_weight_max,time_weight_mean,time_weight_degenerate,train_start_time,train_end_time
inspection_type,,,,,,,,,,,,,,,,
0,64273,111,13289,12,48,88,400,64162,578.036036,24.042380,1.0,2.0,1.571337,False,1970-06-23 03:58:55+00:00,1970-10-05 00:29:00+00:00
1,38900,580,6422,224,56,113,400,38320,66.068966,8.128282,1.0,2.0,1.482090,False,1970-06-23 05:00:03+00:00,1970-10-05 00:29:59+00:00
2,100470,588,7161,27,69,117,400,99882,169.867347,13.033317,1.0,2.0,1.571788,False,1970-06-23 04:01:30+00:00,1970-10-05 00:29:59+00:00
3,100740,648,16252,21,69,109,400,100092,154.462963,12.428313,1.0,2.0,1.580274,False,1970-06-23 04:00:54+00:00,1970-10-05 00:29:59+00:00
4,3813,13,902,73,25,53,400,3800,292.307692,17.097008,1.0,2.0,1.556784,False,1970-06-24 12:16:18+00:00,1970-10-05 00:12:57+00:00


## 10. 최종 Validation에서 공통·타입별 임계값 선택

Test를 사용하지 않고 Validation Recall 99% 이상을 만족하는 후보 중 False Call Reduction이 최대인 임계값을 선택합니다. 동률이면 Recall, 다시 동률이면 threshold가 높은 후보를 선택합니다.

In [11]:
global_threshold_selection = select_threshold(
    validation_df[TARGET], pooled_probability, min_recall=MIN_RECALL
)

thresholds_by_type = {}
type_threshold_rows = []
type_validation_prediction = pd.Series(np.nan, index=validation_df.index, dtype="float64")

for inspection_type in inspection_types:
    type_validation = validation_df.loc[validation_df[TYPE_COLUMN] == inspection_type]
    type_probability = pooled_probability.loc[type_validation.index]
    selection = select_threshold(
        type_validation[TARGET], type_probability, min_recall=MIN_RECALL
    )
    thresholds_by_type[inspection_type] = selection["threshold"]
    selection["inspection_type"] = inspection_type
    type_threshold_rows.append(selection)
    type_validation_prediction.loc[type_validation.index] = (
        type_probability >= selection["threshold"]
    ).astype("int8")

type_threshold_selection = pd.DataFrame(type_threshold_rows).set_index("inspection_type")
global_validation_metrics = pd.Series(
    evaluate_probabilities(
        validation_df[TARGET],
        pooled_probability,
        global_threshold_selection["threshold"],
    ),
    name="global_threshold",
)
type_specific_validation_metrics = pd.Series(
    evaluate_predictions(
        validation_df[TARGET],
        type_validation_prediction,
        pooled_probability,
    ),
    name="type_specific_thresholds",
)

validation_strategy_metrics = pd.DataFrame(
    {
        "fixed_0.5": pooled_metrics,
        "global_threshold": global_validation_metrics,
        "type_specific_thresholds": type_specific_validation_metrics,
    }
).T

threshold_summary = pd.concat(
    [
        pd.DataFrame(
            [{"scope": "global", **global_threshold_selection}]
        ).set_index("scope"),
        type_threshold_selection.rename_axis("scope"),
    ],
    axis=0,
)

display(
    threshold_summary[
        ["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn", "fp", "tn"]
    ]
)
display(
    validation_strategy_metrics[
        ["pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]
    ]
)
logger.info("global_threshold_selection=%s", global_threshold_selection)
logger.info("type_threshold_selection=%s", type_threshold_selection.to_dict(orient="index"))
logger.info("validation_strategy_metrics=%s", validation_strategy_metrics.to_dict(orient="index"))

,threshold,positive_samples,recall,false_call_reduction,tp,fn,fp,tn
scope,,,,,,,,
global,0.000181,357,0.991597,0.283153,354,3,31304,12365
0,0.000121,12,1.000000,0.421104,12,0,7686,5591
1,0.001012,224,0.991071,0.309777,222,2,4278,1920
2,0.001812,27,1.000000,0.539249,27,0,3287,3847
3,0.000287,21,1.000000,0.432814,21,0,9206,7025
4,0.000155,73,1.000000,0.002413,73,0,827,2


,pr_auc,precision,recall,false_call_reduction,f1,tp,fn,fp,tn
fixed_0.5,0.449264,0.273399,0.621849,0.986489,0.379812,222.0,135.0,590.0,43079.0
global_threshold,0.449264,0.011182,0.991597,0.283153,0.022115,354.0,3.0,31304.0,12365.0
type_specific_thresholds,0.449264,0.013846,0.994398,0.421008,0.027312,355.0,2.0,25284.0,18385.0


2026-08-25 03:06:21,993 | INFO | global_threshold_selection={'threshold': 0.00018069491488859057, 'min_recall': 0.99, 'rows': 44026, 'positive_samples': 357, 'tn': 12365, 'fp': 31304, 'fn': 3, 'tp': 354, 'accuracy': 0.288897469677009, 'precision': 0.011182007707372543, 'recall': 0.9915966386554622, 'false_call_reduction': 0.2831528086285466, 'f1': 0.02211463376542246, 'roc_auc': 0.8860953160947909, 'pr_auc': 0.44926386830748244}


2026-08-25 03:06:21,994 | INFO | type_threshold_selection={0: {'threshold': 0.00012145858636358753, 'min_recall': 0.99, 'rows': 13289, 'positive_samples': 12, 'tn': 5591, 'fp': 7686, 'fn': 0, 'tp': 12, 'accuracy': 0.42162690947400105, 'precision': 0.001558846453624318, 'recall': 1.0, 'false_call_reduction': 0.4211041650975371, 'f1': 0.0031128404669260703, 'roc_auc': 0.8725427430895533, 'pr_auc': 0.006494979475133576}, 1: {'threshold': 0.0010115972254425287, 'min_recall': 0.99, 'rows': 6422, 'positive_samples': 224, 'tn': 1920, 'fp': 4278, 'fn': 2, 'tp': 222, 'accuracy': 0.3335409529741514, 'precision': 0.04933333333333333, 'recall': 0.9910714285714286, 'false_call_reduction': 0.30977734753146174, 'f1': 0.09398814563928874, 'roc_auc': 0.9584795498547919, 'pr_auc': 0.706345847439895}, 2: {'threshold': 0.0018124595517292619, 'min_recall': 0.99, 'rows': 7161, 'positive_samples': 27, 'tn': 3847, 'fp': 3287, 'fn': 0, 'tp': 27, 'accuracy': 0.5409858958246055, 'precision': 0.008147254073627036

2026-08-25 03:06:21,994 | INFO | validation_strategy_metrics={'fixed_0.5': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 43079.0, 'fp': 590.0, 'fn': 135.0, 'tp': 222.0, 'accuracy': 0.9835324580929451, 'precision': 0.2733990147783251, 'recall': 0.6218487394957983, 'false_call_reduction': 0.9864892715656415, 'f1': 0.3798118049615056, 'roc_auc': 0.8860953160947909, 'pr_auc': 0.44926386830748244}, 'global_threshold': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 12365.0, 'fp': 31304.0, 'fn': 3.0, 'tp': 354.0, 'accuracy': 0.288897469677009, 'precision': 0.011182007707372543, 'recall': 0.9915966386554622, 'false_call_reduction': 0.2831528086285466, 'f1': 0.02211463376542246, 'roc_auc': 0.8860953160947909, 'pr_auc': 0.44926386830748244}, 'type_specific_thresholds': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 18385.0, 'fp': 25284.0, 'fn': 2.0, 'tp': 355.0, 'accuracy': 0.4256575659837369, 'precision': 0.013846093841413472, 'recall': 0.9943977591036415, 'false_call_reduction': 0

## 11. 고정 모델의 최종 Test 추론

Validation 결과를 확인한 뒤 모델·피처·파라미터와 Validation에서 선택한 threshold를 변경하지 않고 마지막 20% Test를 한 번 추론합니다.

In [12]:
test_probability = pd.Series(np.nan, index=test_df.index, dtype="float64")
type_test_metric_rows = []

for inspection_type in inspection_types:
    feature_columns = feature_columns_by_type[inspection_type]
    type_test = test_df.loc[test_df[TYPE_COLUMN] == inspection_type]
    preprocessor = preprocessors_by_type[inspection_type]
    model = models_by_type[inspection_type]

    X_test = preprocessor.transform(type_test[feature_columns])
    probability = model.predict_proba(X_test)[:, 1]
    test_probability.loc[type_test.index] = probability

    metrics = evaluate_probabilities(type_test[TARGET], probability)
    metrics["inspection_type"] = inspection_type
    type_test_metric_rows.append(metrics)
    logger.info(
        "test_type_metrics type=%d pr_auc=%.6f recall=%.6f fcr=%.6f tp=%d fn=%d",
        inspection_type,
        metrics["pr_auc"],
        metrics["recall"],
        metrics["false_call_reduction"],
        metrics["tp"],
        metrics["fn"],
    )
    del X_test, probability
    gc.collect()

assert test_probability.notna().all()
test_metrics = pd.Series(
    evaluate_probabilities(test_df[TARGET], test_probability),
    name="type_expert_test",
)
type_test_metrics = pd.DataFrame(type_test_metric_rows).set_index("inspection_type")
type_test_metrics[count_columns] = type_test_metrics[count_columns].astype("int64")
fixed_test_metrics = test_metrics.copy()
fixed_test_metrics.name = "fixed_0.5"
global_test_metrics = pd.Series(
    evaluate_probabilities(
        test_df[TARGET],
        test_probability,
        global_threshold_selection["threshold"],
    ),
    name="global_threshold",
)
type_test_prediction = pd.Series(np.nan, index=test_df.index, dtype="float64")
type_selected_test_rows = []
for inspection_type in inspection_types:
    type_test = test_df.loc[test_df[TYPE_COLUMN] == inspection_type]
    type_probability = test_probability.loc[type_test.index]
    threshold = thresholds_by_type[inspection_type]
    type_prediction = (type_probability >= threshold).astype("int8")
    type_test_prediction.loc[type_test.index] = type_prediction
    metrics = evaluate_predictions(type_test[TARGET], type_prediction, type_probability)
    metrics.update({"inspection_type": inspection_type, "threshold": threshold})
    type_selected_test_rows.append(metrics)

type_specific_test_metrics = pd.Series(
    evaluate_predictions(test_df[TARGET], type_test_prediction, test_probability),
    name="type_specific_thresholds",
)
test_strategy_metrics = pd.DataFrame(
    {
        "fixed_0.5": fixed_test_metrics,
        "global_threshold": global_test_metrics,
        "type_specific_thresholds": type_specific_test_metrics,
    }
).T
type_selected_test_metrics = pd.DataFrame(type_selected_test_rows).set_index("inspection_type")

logger.info("pooled_test_metrics_fixed_0.5=%s", fixed_test_metrics.to_dict())
logger.info("test_strategy_metrics=%s", test_strategy_metrics.to_dict(orient="index"))

display(
    test_strategy_metrics[
        ["pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]
    ]
)
display(
    type_selected_test_metrics[
        ["threshold", "positive_samples", "pr_auc", "precision", "recall", "false_call_reduction", "tp", "fn", "fp", "tn"]
    ]
)
display(fixed_test_metrics)
display(
    type_test_metrics[
        [
            "rows",
            "positive_samples",
            "pr_auc",
            "roc_auc",
            "accuracy",
            "precision",
            "recall",
            "false_call_reduction",
            "f1",
            "tp",
            "fn",
            "fp",
            "tn",
        ]
    ]
)

2026-08-25 03:06:22,064 | INFO | test_type_metrics type=0 pr_auc=0.036233 recall=0.015385 fcr=0.998704 tp=3 fn=192


2026-08-25 03:06:22,121 | INFO | test_type_metrics type=1 pr_auc=0.462430 recall=0.695090 fcr=0.835536 tp=538 fn=236


2026-08-25 03:06:22,197 | INFO | test_type_metrics type=2 pr_auc=0.633813 recall=0.515732 fcr=0.996113 tp=377 fn=354


2026-08-25 03:06:22,310 | INFO | test_type_metrics type=3 pr_auc=0.407876 recall=0.369281 fcr=0.994785 tp=226 fn=386


2026-08-25 03:06:22,338 | INFO | test_type_metrics type=4 pr_auc=0.030408 recall=0.000000 fcr=0.972028 tp=0 fn=13


...

2026-08-25 03:06:22,692 | INFO | test_strategy_metrics={'fixed_0.5': {'rows': 88052.0, 'positive_samples': 2325.0, 'tn': 83522.0, 'fp': 2205.0, 'fn': 1181.0, 'tp': 1144.0, 'accuracy': 0.9615454504156634, 'precision': 0.3415945058226336, 'recall': 0.4920430107526882, 'false_call_reduction': 0.9742788153090625, 'f1': 0.4032428621783574, 'roc_auc': 0.8759114949920421, 'pr_auc': 0.38467203647556497}, 'global_threshold': {'rows': 88052.0, 'positive_samples': 2325.0, 'tn': 16331.0, 'fp': 69396.0, 'fn': 93.0, 'tp': 2232.0, 'accuracy': 0.21081860718666243, 'precision': 0.03116099849220975, 'recall': 0.96, 'false_call_reduction': 0.1905000758220864, 'f1': 0.060362662772301325, 'roc_auc': 0.8759114949920421, 'pr_auc': 0.38467203647556497}, 'type_specific_thresholds': {'rows': 88052.0, 'positive_samples': 2325.0, 'tn': 27175.0, 'fp': 58552.0, 'fn': 132.0, 'tp': 2193.0, 'accuracy': 0.33353018670785445, 'precision': 0.036101736768458306, 'recall': 0.9432258064516129, 'false_call_reduction': 0.31699

,pr_auc,precision,recall,false_call_reduction,f1,tp,fn,fp,tn
fixed_0.5,0.384672,0.341595,0.492043,0.974279,0.403243,1144.0,1181.0,2205.0,83522.0
global_threshold,0.384672,0.031161,0.960000,0.190500,0.060363,2232.0,93.0,69396.0,16331.0
type_specific_thresholds,0.384672,0.036102,0.943226,0.316995,0.069542,2193.0,132.0,58552.0,27175.0


,threshold,positive_samples,pr_auc,precision,recall,false_call_reduction,tp,fn,fp,tn
inspection_type,,,,,,,,,,
0,0.000121,195,0.036233,0.012595,0.841026,0.333696,164,31,12857,6439
1,0.001012,774,0.462430,0.078321,0.985788,0.224410,763,11,8979,2598
2,0.001812,731,0.633813,0.059029,0.926129,0.455280,677,54,10792,9020
3,0.000287,612,0.407876,0.022339,0.941176,0.265622,576,36,25209,9118
4,0.000155,13,0.030408,0.017857,1.000000,0.000000,13,0,715,0


rows                    88052.000000
positive_samples         2325.000000
tn                      83522.000000
fp                       2205.000000
fn                       1181.000000
tp                       1144.000000
accuracy                    0.961545
precision                   0.341595
recall                      0.492043
false_call_reduction        0.974279
f1                          0.403243
roc_auc                     0.875911
pr_auc                      0.384672
Name: fixed_0.5, dtype: float64

,rows,positive_samples,pr_auc,roc_auc,accuracy,precision,recall,false_call_reduction,f1,tp,fn,fp,tn
inspection_type,,,,,,,,,,,,,
0,19491,195,0.036233,0.702072,0.988867,0.107143,0.015385,0.998704,0.026906,3,192,25,19271
1,12351,774,0.462430,0.875210,0.826735,0.220311,0.695090,0.835536,0.334577,538,236,1904,9673
2,20543,731,0.633813,0.895394,0.979020,0.830396,0.515732,0.996113,0.636287,377,354,77,19735
3,34939,612,0.407876,0.869764,0.983829,0.558025,0.369281,0.994785,0.444444,226,386,179,34148
4,728,13,0.030408,0.541151,0.954670,0.000000,0.000000,0.972028,0.000000,0,13,20,695


## 12. 원본 무결성과 종료 확인

실행 전후 `dataset.csv`와 `mapping.json`의 SHA-256가 같은지 확인합니다.


In [13]:
DATA_SHA256_AFTER = sha256_file(DATA_PATH)
MAPPING_SHA256_AFTER = sha256_file(MAPPING_PATH)
assert DATA_SHA256_AFTER == DATA_SHA256_BEFORE
assert MAPPING_SHA256_AFTER == MAPPING_SHA256_BEFORE

verification = pd.Series(
    {
        "dataset_sha256_unchanged": True,
        "mapping_sha256_unchanged": True,
        "type_models_trained": len(models_by_type),
        "test_evaluated_once": True,
        "fixed_threshold": DECISION_THRESHOLD,
        "global_threshold": global_threshold_selection["threshold"],
        "type_thresholds": thresholds_by_type,
        "class_weighting": "scale_pos_weight=sqrt(negative/positive)_by_type_train_only",
        "time_weighting": f"linear_{TIME_WEIGHT_MIN:.1f}_to_{TIME_WEIGHT_MAX:.1f}_by_type_train_time",
        "final_scale_pos_weight_raw_by_type": training_summary["scale_pos_weight_raw"].to_dict(),
        "final_scale_pos_weight_sqrt_by_type": training_summary["scale_pos_weight_sqrt"].to_dict(),
        "final_time_weight_mean_by_type": training_summary["time_weight_mean"].to_dict(),
        "log_file": f"docs/peace/{LOG_PATH.name}",
    },
    name="verification",
)
display(verification)
logger.info(
    "source_integrity=PASS test_evaluated_once=True fixed_threshold=%.2f global_threshold=%.8f",
    DECISION_THRESHOLD,
    global_threshold_selection["threshold"],
)
logger.info("experiment_complete=%s", EXPERIMENT_ID)
for handler in logger.handlers:
    handler.flush()


dataset_sha256_unchanged                                                            True
mapping_sha256_unchanged                                                            True
type_models_trained                                                                    5
test_evaluated_once                                                                 True
fixed_threshold                                                                      0.5
global_threshold                                                                0.000181
type_thresholds                        {0: 0.00012145858636358753, 1: 0.0010115972254...
class_weighting                        scale_pos_weight=sqrt(negative/positive)_by_ty...
time_weighting                                      linear_1.0_to_2.0_by_type_train_time
final_scale_pos_weight_raw_by_type     {0: 578.0360360360361, 1: 66.06896551724138, 2...
final_scale_pos_weight_sqrt_by_type    {0: 24.042379999410127, 1: 8.128281830574121, ...
final_time_weight_mea

2026-08-25 03:06:22,894 | INFO | source_integrity=PASS test_evaluated_once=True fixed_threshold=0.50 global_threshold=0.00018069


2026-08-25 03:06:22,895 | INFO | experiment_complete=0825_peace_010_type_expert_sqrt_class_time_weight


## 13. 결론과 다음 실험

이 노트북은 `0825_peace_004_type_expert_walk_forward`의 시간 분할·피처·XGBoost 파라미터·임계값 선택 규칙을 유지한 채, 각 타입의 현재 Train에서 `scale_pos_weight = sqrt(음성 수 / 양성 수)`를 다시 계산하고 동시에 시간 가중치 `sample_weight 1.0→2.0`를 함께 적용한 실험입니다.

- Fold별 raw class ratio는 60.44~882.66, 적용된 `sqrt(scale_pos_weight)`는 7.77~29.71 범위였고, 최종 0~70% 학습에서는 타입 0~4가 각각 raw 578.04 / 66.07 / 169.87 / 154.46 / 292.31, sqrt 24.04 / 8.13 / 13.03 / 12.43 / 17.10이었습니다.
- 모든 Fold와 최종 학습에서 시간 가중치 `sample_weight`는 1.0~2.0 범위로 생성됐고, 저장된 요약에서 `time_weight_degenerate=False`가 유지됐습니다.
- Walk-forward 공통 임계값의 미래 Recall은 100.0% / 96.1% / 100.0%였고 평균 Recall 98.7%, 최저 Recall 96.1%, 평균 False Call Reduction 17.9%였습니다.
- Walk-forward 타입별 임계값의 미래 Recall은 97.9% / 83.6% / 94.9%였고 평균 Recall 92.1%, 최저 Recall 83.6%, 평균 False Call Reduction 38.8%였습니다.
- 최종 Validation에서는 고정 0.5가 Recall 62.2% / FCR 98.6%, 공통 임계값이 Recall 99.16% / FCR 28.32%, 타입별 임계값이 Recall 99.44% / FCR 42.10%를 기록했습니다.
- 최종 Test에서는 고정 0.5가 PR-AUC 0.384672, Recall 49.2%, FCR 97.4%였고, 공통 임계값은 Recall 96.0% / FCR 19.1%, 타입별 임계값은 Recall 94.3% / FCR 31.7%였습니다.
- `007`의 강한 클래스 가중치보다 score 분포 왜곡이 완화돼 Test PR-AUC와 임계값 전략의 FCR이 개선됐지만, `008` 시간 가중치 단독보다 공통 임계값 FCR은 여전히 낮습니다.
- 따라서 `sqrt(scale_pos_weight) + 시간 가중치`는 클래스 가중치 강도를 줄이는 방향이 맞다는 근거는 제공했지만, Recall 99% 제약 운영에서는 아직 FP 비용이 커서 시간 가중치 단독(`008`)과의 비교를 기준으로 채택 여부를 판단하는 편이 안전합니다.
